# Kitsune-TTS · Add your own speakers

A small Colab launcher for `finetune.py`: configure your data, run training, and download your model.

*Fine-tuning is experimental; defaults may evolve as we test more voices.*

- Select **Runtime → Change runtime type → T4 GPU**.
- Use Brazilian Portuguese recordings and accurate transcripts you have permission to use.
- The base checkpoint stays untouched. Full fine-tuning prioritizes your new voices; old voices may change in the specialized model.
- Same ~39M architecture, no adapters, no quantization. Runtime and quality depend on your dataset and GPU.

Training code lives in the repository, not in notebook cells. [Full guide](https://github.com/Heitorkk2/Kitsune-TTS/blob/main/examples/finetune/README.md) · [V1 weights](https://huggingface.co/Heitorkk2/Kitsune-TTS-V1) · GPL-3.0


## 1 · Install

The source is cloned only if the checkout does not exist. Choose a branch/tag in `GITHUB_REF`; pin a release for reproducible runs. Existing checkouts are not replaced.

If the Hugging Face model is still private, authenticate with `huggingface_hub.login()` in a separate cell before training. Never put a token into a shared notebook.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_BASE = Path("/content/Kitsune-TTS")
GITHUB_REF = "main"  # Branch or release tag.
if not REPO_BASE.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", GITHUB_REF,
        "https://github.com/Heitorkk2/Kitsune-TTS.git", str(REPO_BASE),
    ], check=True)
if not (REPO_BASE / "finetune.py").is_file():
    raise RuntimeError("This checkout needs the config-driven finetune.py. Use an updated checkout.")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_BASE / "requirements.txt"),
], check=True)
if str(REPO_BASE) not in sys.path:
    sys.path.insert(0, str(REPO_BASE))


## 2 · Configure your voices

Upload your dataset, then edit the paths below. Each speaker needs a WAV directory and a UTF-8 transcript with `filename.wav|text` lines:

```text
0001.wav|Olá, esta é uma frase de exemplo.
0002.wav|Hoje o céu está muito bonito.
```

Existing `filename|speaker_id|language|text` lines also work: the first/last fields are used and new IDs are assigned automatically. Do not put `|` in the text. WAVs are converted to mono PCM16 at the model's sample rate when needed; source recordings are preserved.

Add a dictionary to `speakers` for each voice. Min/max batches must be divisible by the speaker count. Other settings use the tested defaults from the Python code. `/content` is temporary: download outputs or use an already-mounted Drive folder.


In [ ]:
RUN_CONFIG = {
    "run_name": "my_voice_run",
    "output_root": "/content/kitsune_finetune",
    "model": {
        "repo_id": "Heitorkk2/Kitsune-TTS-V1",
        "revision": "main",
        "local_dir": None,  # Or a folder with latest_model_fp16.pth + model_config.json.
    },
    "speakers": [
        {
            "name": "my_speaker",
            "wav_dir": "/content/voice_data/my_speaker/wavs",
            "transcript": "/content/voice_data/my_speaker/train.txt",
        },
        # Add another speaker dictionary here if needed.
    ],
    "dataset_zip": None,  # Optional uploaded ZIP; existing dataset_root is not replaced.
    "dataset_root": "/content/voice_data",
    "resume_state": None,  # Path to training/specialized_latest.pth to continue a run.
    "preview_text": "Olá! Esta é uma demonstração da minha nova voz.",
    "training": {
        "max_steps": 2400,
        "warmup_steps": 100,
        "batch_size": 16,  # Maximum physical batch; long clips use smaller batches.
        "min_batch_size": 4,
        "reference_batch_size": 8,  # Lower this and min_batch_size if VRAM is insufficient.
        "grad_accum_steps": 2,
        "embedding_lr": 1e-3,
        "model_lr": 2e-5,
        "precision": "fp16",
        "save_interval": 200,
    },
}


In [ ]:
from kitsune.finetune.config import FineTuneConfig

CONFIG_PATH = Path("/content/finetune_run.json")
config = FineTuneConfig.from_dict(RUN_CONFIG, base_dir=CONFIG_PATH.parent)
CONFIG_PATH.write_text(
    json.dumps(config.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8",
)
print("Configuration:", CONFIG_PATH)
print("Outputs:", config.run_dir)
print("Speakers:", [speaker.name for speaker in config.speakers])
print("Optimizer updates:", config.training.max_steps)


## 3 · Run

This command downloads the FP16 base and config, prepares data/cache, trains, verifies the base file, and exports results. The first pass builds the cache.

The recipe is unchanged: new rows start at zero, embedding-only warm-up, then full-generator training with mel + KL + duration losses and embedding regularizers. AMP uses FP16 computation with FP32 parameters/optimizer state. No discriminator is trained.

Previews and a resumable checkpoint are saved every `save_interval` updates. Training loss is not a held-out speaker-similarity score. Accumulation does not inherently make training faster. There is no guaranteed three-hour result.


In [ ]:
subprocess.run([
    sys.executable, "-u", str(REPO_BASE / "finetune.py"),
    "--config", str(CONFIG_PATH),
], cwd=REPO_BASE, check=True)


## 4 · Listen

Check identity, pronunciation and artifacts. During training, previews also appear under `training/samples/` in your run directory. The cells below use the final export.


In [ ]:
from IPython.display import Audio, display

FINAL = config.run_dir / "final"
for speaker in config.speakers:
    sample = FINAL / "samples" / f"{speaker.name}.wav"
    if not sample.is_file():
        raise FileNotFoundError(f"No final preview yet: {sample}")
    print(speaker.name)
    display(Audio(str(sample)))


## 5 · Download and resume

The ZIP includes `latest_model_fp16.pth`, matching `model_config.json`, the manifest, run configuration and preview WAVs. It loads directly with the existing Python runtime.

**Download `training/specialized_latest.pth` separately if you want to resume**; optimizer state is intentionally excluded from the inference ZIP. Also keep your original dataset. Nothing is uploaded automatically.


In [ ]:
from google.colab import files

ARCHIVE = config.run_dir / f"{config.run_name}_specialized.zip"
files.download(str(ARCHIVE))
print("Keep this separate resume file:", config.run_dir / "training/specialized_latest.pth")


To resume, restore the same dataset/base revision and speaker names/order, set `resume_state`, increase `training.max_steps` beyond the saved update, and rerun configuration + training. Resume states from the previous public notebook are also accepted. FP16 rounding and recreated data-loader/RNG state mean continuation is not bit-exact.

For local training, use the same JSON (update Colab paths first):

```bash
python finetune.py --config /path/to/finetune_run.json
```

For inference, point `KitsuneSynthesizer` at the exported checkpoint and its config. Keep the original V1 weights for original voices. See the [export guide](https://github.com/Heitorkk2/Kitsune-TTS/blob/main/examples/export/README.md) for ONNX.
